In [ ]:
import sys
sys.path.insert(1, '/media/bruno/Matosak/repos/SenForFlood')

import torch
from models import AttUNet_t
import numpy as np
import matplotlib.pyplot as plt
from SenForFlood import SenForFlood

In [ ]:
data = torch.load('/media/bruno/Matosak/repos/SenForFlood/Examples/models_FM/world_b64/Checkpoints/model-e0091.pt', weights_only=True)

In [ ]:
model = AttUNet_t(2,2,64,0.1, use_terrain=True).to('cuda')
model.load_state_dict(data['model_state_dict'])
model.eval()

In [ ]:
test_dataset = SenForFlood(
    dataset_folder='/media/bruno/Matosak/SenForFlood',
    chip_size=512,
    events=['DFO_4459_Bangladesh'],
    data_to_include=['s1_before_flood', 's1_during_flood'],
    use_data_augmentation=True
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=2, drop_last=False)

In [ ]:
for ind, (x1, x0) in enumerate(test_loader):
    x1 = x1[:,:2]
    x0 = x0[:,:2]
    break

model.eval()
with torch.no_grad():
    xts = [x0]
    t_span = torch.linspace(0, 1, 10)
    for s,t in zip(t_span[:-1], t_span[1:]):
        xt = xts[-1]
        t_expanded = (torch.zeros([xt.shape[0], 1, xt.shape[2], xt.shape[3]])+t)
        xts.append((model(xt.to('cuda'), t_expanded.to('cuda')).detach().cpu() * (t - s) + xt).detach().cpu())
    # plot_pairs(x0, x1, xts, f'models_FM/{model_id}/plots')

In [ ]:
xts[-1].shape

In [ ]:
for i in range(10):
    plt.imshow(xts[i][-1][0], vmin=0, vmax=1)
    plt.colorbar()
    plt.show()

In [ ]:
from FM_train import plot_loss
import torch

data = torch.load('/media/bruno/Matosak/repos/SenForFlood/Examples/models_FM/test_bangladesh/Checkpoints/model-e0004.pt', weights_only=True)
plot_loss(data['loss'])

In [ ]:
import pickle
import numpy as np

with open('/media/bruno/Matosak/repos/SenForFlood/percentile_limits_copy.pickle', 'rb') as f:
    data = pickle.load(f)

data

In [ ]:
for key1 in data.keys():
    for i in range(len(data[key1])):
        for key2 in data[key1][i].keys():
            if np.isnan(data[key1][i][key2]):
                print(key1, i, key2, data[key1][i][key2])

In [ ]:
np.isnan(data['s1_before_flood'][2]['0'])

In [ ]:
import sys
sys.path.insert(1, '/media/bruno/Matosak/repos/SenForFlood')
from SenForFlood import SenForFlood
import matplotlib.pyplot as plt
import numpy as np

dataset = SenForFlood(
    dataset_folder='/media/bruno/Matosak/SenForFlood',
    chip_size=512,
    events=['DFO_4459_Bangladesh'],
    data_to_include=['s1_before_flood', 's1_during_flood'],
    use_data_augmentation=False,
    scale_0_1=False
)

def return_image_data(d1, d2):
    data_1 = d1.copy()
    data_2 = d2.copy()

    div_1 = data_1[0]/data_1[1]
    div_2 = data_2[0]/data_2[1]

    data_1 = np.concat([data_1, div_1[None,:,:]], axis=0)
    data_2 = np.concat([data_2, div_2[None,:,:]], axis=0)

    p = np.percentile(np.concatenate([data_1, data_2], axis=1), q=[2,98], axis=(1,2))
    for i in range(data_1.shape[0]):
        data_1[i] = (data_1[i]-p[0][i])/(p[1][i]-p[0][i])
        data_2[i] = (data_2[i]-p[0][i])/(p[1][i]-p[0][i])
    data_1 = np.clip(data_1, 0, 1)
    data_2 = np.clip(data_2, 0, 1)

    return np.moveaxis(data_1, 0, -1), np.moveaxis(data_2, 0, -1)

for ind in range(0,5,1):

    data_before = dataset[ind][0][:2]
    data_after = dataset[ind][1][:2]

    f, ax = plt.subplots(2, 2, figsize=(7,6))

    d1, d2 = return_image_data(data_before, data_after)
    ax[0,0].imshow(d1)
    ax[0,0].axis('off')
    ax[0,0].title.set_text('$X_1$')

    ax[0,1].imshow(d2)
    ax[0,1].axis('off')
    ax[0,1].title.set_text('$X_0$')

    diff = np.linalg.norm(data_after-data_before, axis=0)
    im = ax[1,0].imshow(diff, vmin=np.percentile(diff, 2), vmax=np.percentile(diff, 98))
    # plt.colorbar(im, ax=ax[1,0])
    ax[1,0].axis('off')
    ax[1,0].title.set_text('$abs(X_1-X_0)$')

    diff_small = diff.copy()
    diff_small[diff_small>=np.percentile(diff, 30)] = None
    diff_large = diff.copy()
    diff_large[diff_large<=np.percentile(diff, 70)] = None
    ax[1,1].imshow(diff_small, cmap='Greens', vmin=-1000, vmax=1000)
    ax[1,1].imshow(diff_large, cmap='Blues', vmin=-1000, vmax=0)
    ax[1,1].axis('off')
    ax[1,1].title.set_text('$<p_{30}$ (green), $>p_{70}$ (blue)')

    plt.tight_layout()
    plt.show()